In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)

from datasets import load_dataset
import random
from nnsight import LanguageModel
import torch as t
from torch import nn
from attribution import patching_effect, Submodule
from dictionary_learning import AutoEncoder, ActivationBuffer, JumpReluAutoEncoder
from dictionary_learning.dictionary import IdentityDict
from dictionary_loading_utils import load_saes_and_submodules, load_gemma_transcoders_and_submodules
from dictionary_learning.utils import hf_dataset_to_generator
from transformers import BitsAndBytesConfig
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm import tqdm
from typing import Literal
import gc
from collections import defaultdict
import hashlib


DEBUGGING = False

if DEBUGGING:
    tracer_kwargs = dict(scan=True, validate=True)
else:
    tracer_kwargs = dict(scan=False, validate=False)

# model hyperparameters
DTYPE = t.bfloat16
DEVICE = 'cuda:0'
model = LanguageModel('google/gemma-2-2b', device_map=DEVICE, dispatch=True,
                      attn_implementation="eager", torch_dtype=DTYPE)
activation_dim = 2304

/workspace/envs/sae-circuits/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:52<00:00, 17.65s/it]


In [2]:
# dataset hyperparameters
dataset = load_dataset("LabHC/bias_in_bios")
profession_dict = {'professor' : 21, 'nurse' : 13}
male_prof = 'professor'
female_prof = 'nurse'

SEED = 42

def get_text_batches(
    split: Literal["train", "test"] = "train",
    ambiguous=True,
    batch_size=32,
    seed=SEED
):
    data = dataset[split]
    if ambiguous:
        neg = [x['hard_text'] for x in data if x['profession'] == profession_dict[male_prof] and x['gender'] == 0]
        pos = [x['hard_text'] for x in data if x['profession'] == profession_dict[female_prof] and x['gender'] == 1]
        n = min([len(neg), len(pos)])
        neg, pos = neg[:n], pos[:n]
        data = neg + pos
        labels = [0]*n + [1]*n
        idxs = list(range(2*n))
        random.Random(seed).shuffle(idxs)
        data, labels = [data[i] for i in idxs], [labels[i] for i in idxs]
        true_labels = spurious_labels = labels
    else:
        neg_neg = [x['hard_text'] for x in data if x['profession'] == profession_dict[male_prof] and x['gender'] == 0]
        neg_pos = [x['hard_text'] for x in data if x['profession'] == profession_dict[male_prof] and x['gender'] == 1]
        pos_neg = [x['hard_text'] for x in data if x['profession'] == profession_dict[female_prof] and x['gender'] == 0]
        pos_pos = [x['hard_text'] for x in data if x['profession'] == profession_dict[female_prof] and x['gender'] == 1]
        n = min([len(neg_neg), len(neg_pos), len(pos_neg), len(pos_pos)])
        neg_neg, neg_pos, pos_neg, pos_pos = neg_neg[:n], neg_pos[:n], pos_neg[:n], pos_pos[:n]
        data = neg_neg + neg_pos + pos_neg + pos_pos
        true_labels     = [0]*n + [0]*n + [1]*n + [1]*n
        spurious_labels = [0]*n + [1]*n + [0]*n + [1]*n
        idxs = list(range(4*n))
        random.Random(seed).shuffle(idxs)
        data, true_labels, spurious_labels = [data[i] for i in idxs], [true_labels[i] for i in idxs], [spurious_labels[i] for i in idxs]

    batches = [
        (data[i:i+batch_size], t.tensor(true_labels[i:i+batch_size], device=DEVICE), t.tensor(spurious_labels[i:i+batch_size], device=DEVICE)) for i in range(0, len(data), batch_size)
    ]
    return batches

def get_subgroups(
        split: Literal["train", "test"] = "test",
        batch_size=32,
):
    data = dataset[split]
    neg_neg = [x['hard_text'] for x in data if x['profession'] == profession_dict[male_prof] and x['gender'] == 0]
    neg_pos = [x['hard_text'] for x in data if x['profession'] == profession_dict[male_prof] and x['gender'] == 1]
    pos_neg = [x['hard_text'] for x in data if x['profession'] == profession_dict[female_prof] and x['gender'] == 0]
    pos_pos = [x['hard_text'] for x in data if x['profession'] == profession_dict[female_prof] and x['gender'] == 1]
    subgroups = [(neg_neg, (0, 0)), (neg_pos, (0, 1)), (pos_neg, (1, 0)), (pos_pos, (1, 1))]

    out = {}
    for data, label_profile in subgroups:
        out[label_profile] = []
        for i in range(0, len(data), batch_size):
            text = data[i:i+batch_size]
            out[label_profile].append((
                text,
                t.tensor([label_profile[0]]*len(text), device=DEVICE),
                t.tensor([label_profile[1]]*len(text), device=DEVICE)
            ))
    return out

In [10]:
def pool_acts(acts, attn_mask):
    return (acts * attn_mask[:, :, None]).sum(1) / attn_mask.sum(1)[:, None]

@t.no_grad()
def collect_activations(model, layer, text_batches):
    with tqdm(total=len(text_batches), desc="Collecting activations") as pbar:
        for text_batch, *labels in text_batches:
            with model.trace(text_batch, **tracer_kwargs):
                attn_mask = model.inputs[1]['attention_mask']
                acts = model.model.layers[layer].output[0]
                pooled_acts = pool_acts(acts, attn_mask).save()
            yield pooled_acts.value, *labels
            pbar.update(1)

In [11]:
layer = 22  # probe layer

class Probe(nn.Module):
    def __init__(self, activation_dim, dtype=DTYPE):
        super().__init__()
        self.net = nn.Linear(activation_dim, 1, bias=True, dtype=dtype)

    def forward(self, x):
        return self.net(x).squeeze(-1)

@t.no_grad()
def test_probe(probe, activation_batches):
    corrects = defaultdict(list)
    for acts, *labels in activation_batches:
        logits = probe(acts)
        preds = (logits > 0.0).long()
        for idx, label in enumerate(labels):
            corrects[idx].append(preds == label)
    return {idx: t.cat(corrects[idx]).float().mean().item() for idx in corrects}

# Load probe trained on ambiguous data (from bib_shift.ipynb)
save_path = f"probe_layer_{layer}_{str(DTYPE).split('.')[-1]}.pt"
assert os.path.exists(save_path), f"Run bib_shift.ipynb first to generate {save_path}"
probe = t.load(save_path, weights_only=False)
print(f"Loaded probe from {save_path}")

Loaded probe from probe_layer_22_bfloat16.pt


In [12]:
# Metric function: probe score, sign-flipped for correct class
def metric_fn(model, labels=None):
    attn_mask = model.inputs[1]['attention_mask']
    acts = model.model.layers[layer].output[0]
    acts = pool_acts(acts, attn_mask)
    return t.where(labels == 0, probe(acts), -probe(acts))

# Transcoder-based ablation

Transcoders (`google/gemma-scope-2b-pt-transcoders`) replace each MLP sublayer:
- **Input**: output of `pre_feedforward_layernorm` (normalized residual stream entering MLP)
- **Output**: approximation of `post_feedforward_layernorm` output (normalized MLP contribution)

They share the same JumpReLU architecture as the GemmaScope SAEs, but instead of reconstructing their input, they *transcode* MLP inputs to MLP outputs through a sparse feature bottleneck.

The ablation procedure:
1. Encode MLP input → transcoder features
2. Zero-ablate selected features
3. Decode back → modified MLP output (keeping the reconstruction residual to preserve unexplained variance)
4. Evaluate probe accuracy

In [7]:
# Load transcoders for layers 0..layer
tc_submodules, tc_dicts = load_gemma_transcoders_and_submodules(
    model,
    thru_layer=layer,
    dtype=DTYPE,
    device=DEVICE,
)

Loading Gemma transcoders: 100%|██████████| 23/23 [04:26<00:00, 11.59s/it]


In [8]:
# tc_dicts
tc_submodules

[TranscoderSubmodule(name='tc_mlp_0', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_1', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_2', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_3', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_4', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_5', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_6', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=

In [ ]:
# Compute transcoder feature effects on the probe metric
# Uses gradient x activation (GradInput) attribution: effect_i = f_i * (d metric / d f_i)
# This correctly handles transcoders where input != output by computing the residual
# as (actual_mlp_output - transcoder_reconstruction) and keeping it fixed.

n_batches = 20
batch_size = 1

tc_effects_dir = "effects/tc"
os.makedirs(tc_effects_dir, exist_ok=True)

for batch_idx, (clean, labels, _) in tqdm(
    enumerate(get_text_batches(split="train", ambiguous=True, batch_size=batch_size)),
    total=n_batches
):
    if batch_idx == n_batches:
        break

    hash_input = clean + [s.name for s in tc_submodules]
    hash_digest = hashlib.md5(''.join(hash_input).encode()).hexdigest()
    cache_path = f"{tc_effects_dir}/{hash_digest}.pt"
    if os.path.exists(cache_path):
        continue

    tc_feature_acts = {}
    tc_grads = {}

    with model.trace(clean, **tracer_kwargs):
        for submod in tc_submodules:
            tc = tc_dicts[submod]
            x_in = submod.get_activation()    # pre_feedforward_ln output
            x_out = submod.get_output()        # actual post_feedforward_ln output
            f = tc.encode(x_in)
            x_hat = tc.decode(f)
            res = (x_out - x_hat).detach()    # freeze residual so grad flows through f only
            submod.set_activation(x_hat + res)

            # Save f and register grad capture BEFORE backward.
            # Pattern from attribution.py: saved_proxy.grad.save() must be called
            # before backward() so nnsight installs the gradient hook in time.
            tc_feature_acts[submod] = f.save()
            tc_grads[submod] = tc_feature_acts[submod].grad.save()

        metric = metric_fn(model, labels=labels).sum()
        metric.backward()

    # All .save() proxies are now resolved; compute GradInput and move to CPU
    to_save = {
        submod.name: (tc_feature_acts[submod].value * tc_grads[submod].value).detach().cpu()
        for submod in tc_submodules
    }

    t.save(to_save, cache_path)
    del tc_feature_acts, tc_grads
    gc.collect()

100%|██████████| 25/25 [00:15<00:00,  1.66it/s]


In [18]:
# Aggregate transcoder effects across batches
aggregated_tc_effects = {submod.name: 0 for submod in tc_submodules}

for idx, (clean, *_) in enumerate(
    get_text_batches(split="train", ambiguous=True, batch_size=batch_size)
):
    if idx == n_batches:
        break
    hash_input = clean + [s.name for s in tc_submodules]
    hash_digest = hashlib.md5(''.join(hash_input).encode()).hexdigest()
    effects = t.load(f"{tc_effects_dir}/{hash_digest}.pt")
    for submod in tc_submodules:
        # Sum over sequence (excluding BOS), average over batch
        aggregated_tc_effects[submod.name] += effects[submod.name][:, 1:, :].sum(dim=1).mean(dim=0)

aggregated_tc_effects = {k: v / (batch_size * n_batches) for k, v in aggregated_tc_effects.items()}

In [19]:
# Display top transcoder features per layer
threshold = 6.0
count = 0
for submod in tc_submodules:
    v = aggregated_tc_effects[submod.name]
    top_idxs = (v > threshold).nonzero()
    if len(top_idxs) > 0:
        print(submod.name)
        for idx in top_idxs:
            print(f"  {idx.item()} : {v[idx].item():.4f}")
            count += 1
print(f"\ntotal features above threshold: {count}")

tc_mlp_0
  127 : 7.5312
  427 : 10.1250
  638 : 8.3125
  1966 : 11.5000
  3066 : 6.1875
  3406 : 7.2500
  5123 : 13.1875
  5413 : 12.3750
  5441 : 11.1875
  6051 : 9.5000
  6664 : 14.6250
  7403 : 12.7500
  7745 : 6.3750
  8240 : 12.3750
  8773 : 7.0938
  8909 : 12.9375
  9026 : 8.5000
  10046 : 7.8125
  10322 : 6.5938
  10382 : 6.5938
  11132 : 6.9062
  11569 : 6.4688
  11685 : 14.8125
  12677 : 7.3750
  12754 : 6.2812
  13004 : 24.5000
  13840 : 7.9688
  14474 : 22.7500
  14963 : 21.2500
  15498 : 19.3750
  16031 : 13.2500

total features above threshold: 31


## Identify features to ablate

Inspect top features on [Neuronpedia](https://neuronpedia.org) to determine which are gender-related (spurious) vs profession-related (causal).
Add the gender-related feature indices to `tc_feats_to_ablate` below.

In [26]:
# Helper: view a transcoder feature on Neuronpedia
# Note: Neuronpedia uses the SAE release naming; transcoder features may not have entries yet.
# Use the SAE dashboard for the same layer as a proxy for feature interpretation.
from IPython.display import IFrame

def view_tc_feature(layer_idx, feature_idx):
    # Approximate using the MLP SAE at the same layer for interpretation
    sae_id = f"{layer_idx}-gemmascope-transcoder-16k"
    url = f"https://neuronpedia.org/gemma-2-2b/{sae_id}/{feature_idx}?embed=true&embedexplanation=true&embedplots=true"
    return IFrame(url, width=1200, height=600)

# Example: view feature 0 of layer 0
view_tc_feature(22, 8768)

In [27]:
tc_submodules

[TranscoderSubmodule(name='tc_mlp_0', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_1', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_2', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_3', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_4', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_5', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06)),
 TranscoderSubmodule(name='tc_mlp_6', pre_feedforward_ln=Gemma2RMSNorm((2304,), eps=1e-06), post_feedforward_ln=

In [28]:
# Fill in transcoder features judged as gender-related (spurious) after inspecting above.
# Structure mirrors bib_shift.ipynb: keys are tc_submodule names, values are feature indices.
tc_feats_to_ablate_raw = {
    'tc_mlp_0':[],
    'tc_mlp_1':[],
    'tc_mlp_2':[],
    'tc_mlp_3':[],
    'tc_mlp_4':[],
    'tc_mlp_5':[],
    'tc_mlp_6':[],
    'tc_mlp_7':[],
    'tc_mlp_8':[],
    'tc_mlp_9':[],
    'tc_mlp_10':[],
    'tc_mlp_11':[],
    'tc_mlp_12':[],
    'tc_mlp_13':[],
    'tc_mlp_14':[],
    'tc_mlp_15':[],
    'tc_mlp_16':[],
    'tc_mlp_17':[],
    'tc_mlp_18':[],
    'tc_mlp_19':[],
    'tc_mlp_20':[8198, 9029],
    'tc_mlp_21':[8768, 1594],
    'tc_mlp_22':[11607],
}   

# Example (replace with your annotated indices):
# tc_feats_to_ablate_raw["tc_mlp_5"] = [1234, 5678]

print(f"total features to ablate: {sum(len(v) for v in tc_feats_to_ablate_raw.values())}")

total features to ablate: 5


In [29]:
# Convert to boolean n-hot tensors keyed by submodule
def n_hot(feats, dim):
    out = t.zeros(dim, dtype=t.bool, device=DEVICE)
    for feat in feats:
        out[feat] = True
    return out

tc_feats_to_ablate = {
    submod: n_hot(tc_feats_to_ablate_raw[submod.name], tc_dicts[submod].dict_size)
    for submod in tc_submodules
}

In [32]:
@t.no_grad()
def collect_acts_ablated_tc(
    text_batches,
    model,
    tc_submodules,
    tc_dicts,
    to_ablate,   # dict: TranscoderSubmodule -> bool tensor of features to zero
    layer,
):
    """
    Collect mean-pooled layer activations with transcoder feature ablation.

    For each MLP layer:
      1. Encode MLP input with the transcoder
      2. Zero selected features
      3. Decode back and add the reconstruction residual (actual_output - reconstruct)
      4. Write the result back as the MLP output

    The residual ensures that ablating features only removes their specific contribution
    rather than introducing uncontrolled shifts from transcoder reconstruction error.
    """
    with tqdm(total=len(text_batches), desc="Collecting activations (TC ablation)") as pbar:
        for text, *labels in text_batches:
            with model.trace(text, **tracer_kwargs):
                for submod in tc_submodules:
                    tc = tc_dicts[submod]
                    feat_idxs = to_ablate[submod]
                    if not feat_idxs.any():
                        continue
                    x_in = submod.get_activation()   # pre_feedforward_ln output
                    x_out = submod.get_output()       # actual post_feedforward_ln output
                    f = tc.encode(x_in)
                    x_hat = tc.decode(f)
                    res = x_out - x_hat              # reconstruction residual
                    f[..., feat_idxs] = 0.           # zero-ablate selected features
                    submod.set_activation(tc.decode(f) + res)

                attn_mask = model.inputs[1]['attention_mask']
                act = model.model.layers[layer].output[0]
                pooled_act = pool_acts(act, attn_mask).save()

            yield pooled_act.value, *labels
            pbar.update(1)

## Accuracy after ablating transcoder features

In [33]:
ambiguous_accs = test_probe(
    probe,
    activation_batches=collect_acts_ablated_tc(
        get_text_batches(split="test", ambiguous=True),
        model, tc_submodules, tc_dicts, tc_feats_to_ablate, layer
    ),
)
print(f"Ambiguous test accuracy: {ambiguous_accs[0]}")

unambiguous_accs = test_probe(
    probe,
    activation_batches=collect_acts_ablated_tc(
        get_text_batches(split="test", ambiguous=False),
        model, tc_submodules, tc_dicts, tc_feats_to_ablate, layer
    ),
)
print(f"Ground truth accuracy: {unambiguous_accs[0]}")
print(f"Spurious accuracy: {unambiguous_accs[1]}")

for subgroup, batches in get_subgroups().items():
    subgroup_accs = test_probe(
        probe,
        activation_batches=collect_acts_ablated_tc(
            batches,
            model, tc_submodules, tc_dicts, tc_feats_to_ablate, layer
        ),
    )
    print(f"Subgroup {subgroup} accuracy: {subgroup_accs[0]}")

Ambiguous test accuracy: 0.3428206145763397


Ground truth accuracy: 0.27592167258262634
Spurious accuracy: 0.5547235012054443


Subgroup (0, 0) accuracy: 0.4160444140434265


Subgroup (0, 1) accuracy: 0.27607959508895874


Subgroup (1, 0) accuracy: 0.20737327635288239


Subgroup (1, 1) accuracy: 0.2660315930843353


## Skyline: top transcoder features by effect magnitude

In [ ]:
# Identify top-N features automatically (skyline — no human filtering)
skyline_threshold = 6.0
top_tc_feats_raw = {}
total_skyline = 0
for submod in tc_submodules:
    v = aggregated_tc_effects[submod.name]
    idxs = [idx.item() for idx in (v > skyline_threshold).nonzero()]
    top_tc_feats_raw[submod.name] = idxs
    total_skyline += len(idxs)
    if idxs:
        print(f"{submod.name}: {idxs}")
print(f"\ntotal skyline features: {total_skyline}")

top_tc_feats_to_ablate = {
    submod: n_hot(top_tc_feats_raw[submod.name], tc_dicts[submod].dict_size)
    for submod in tc_submodules
}

In [ ]:
# Evaluate skyline (upper bound: ablating all high-effect features regardless of interpretation)
ambiguous_accs = test_probe(
    probe,
    activation_batches=collect_acts_ablated_tc(
        get_text_batches(split="test", ambiguous=True),
        model, tc_submodules, tc_dicts, top_tc_feats_to_ablate, layer
    ),
)
print(f"[Skyline] Ambiguous test accuracy: {ambiguous_accs[0]}")

unambiguous_accs = test_probe(
    probe,
    activation_batches=collect_acts_ablated_tc(
        get_text_batches(split="test", ambiguous=False),
        model, tc_submodules, tc_dicts, top_tc_feats_to_ablate, layer
    ),
)
print(f"[Skyline] Ground truth accuracy: {unambiguous_accs[0]}")
print(f"[Skyline] Spurious accuracy: {unambiguous_accs[1]}")

for subgroup, batches in get_subgroups().items():
    subgroup_accs = test_probe(
        probe,
        activation_batches=collect_acts_ablated_tc(
            batches,
            model, tc_submodules, tc_dicts, top_tc_feats_to_ablate, layer
        ),
    )
    print(f"[Skyline] Subgroup {subgroup} accuracy: {subgroup_accs[0]}")